In [ ]:
!pip install openai-whisper chromadb rank_bm25 sentence-transformers transformers accelerate bitsandbytes --break-system-packages -q

In [ ]:
import re
import json
import torch
import numpy as np
import whisper
import chromadb
import time
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from rank_bm25 import BM25Okapi

In [ ]:
!apt-get install -y espeak espeak-data libespeak1 > /dev/null 2>&1
print("espeak installed")

In [ ]:
import subprocess
from IPython.display import Audio, display
import tempfile
import os

In [ ]:
if torch.cuda.is_available():
    print(f"GPU ready: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHUNKS_DIR  = "/content/drive/MyDrive/BM25"
CHROMA_PATH  = "/content/drive/MyDrive/chroma_db"
GT_PATH = "/content/drive/MyDrive/ground_truth.json"
OUT_PATH = "/content/drive/MyDrive/evaluation_results.json"

In [ ]:
def timer(label, timings_dict):
    class Timer:
        def __enter__(self):
            self.start = time.time()
            return self
        def __exit__(self, *args):
            timings_dict[label] = time.time() - self.start
    return Timer()

def print_timings(timings):
    print(f"\n")
    print("  Latency breakdown")
    print(f"\n")
    total = sum(timings.values())
    for stage, t in timings.items():
        bar   = "█" * int((t / total) * 30) if total > 0 else ""
        print(f"  {stage:<25} {t:.3f}s  {bar}")
    print(f"  {'TOTAL':<25} {total:.3f}s")

In [ ]:
#embedder
embedder = SentenceTransformer("all-mpnet-base-v2")
print("Embedder ready")

#chromaDB
client     = chromadb.PersistentClient(path=CHROMA_PATH)


In [ ]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder ready")

In [ ]:
MODEL_ID   = "mistralai/Mistral-7B-Instruct-v0.2"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading Mistral (first time takes a few minutes)...")
model_llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"Mistral loaded on: {next(model_llm.parameters()).device}")

modello small

In [ ]:
def rerank(query, candidates):
    pairs  = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)
    for i, score in enumerate(scores):
        candidates[i]["rerank_score"] = float(score)
    return sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)

In [ ]:
model_asr = whisper.load_model("small", device="cuda")
print("Whisper ready")

In [ ]:
def build_prompt(query, chunks):
    context_parts = []
    for i, chunk in enumerate(chunks):
        page = chunk["metadata"].get("page_number", "?")
        context_parts.append(f"[Chunk {i+1} | Page {page}]\n{chunk['text']}")
    context = "\n\n".join(context_parts)
    return f"""You are an automotive diagnostic assistant. Answer the technician's question using ONLY the manual excerpts provided below.

Rules:
- Only use information from the provided excerpts
- Always cite the page number in square brackets e.g. [Page 47]
- If the answer is not in the excerpts, say "I could not find this in the manual"
- Be concise and precise

Manual excerpts:
{context}

Technician's question: {query}

Answer:"""

In [ ]:
def retrieve_all(query, top_k=5, timings=None, prefix=""):
    """
    Runs BM25, dense, and RRF fusion separately.
    Returns all three result sets for inspection.
    """
    if collection is None or bm25 is None:
        print("ERROR: no vehicle loaded — call check_vehicle() first")
        return None

    # BM25
    with timer (f"{prefix}BM25",  timings or {}):
      tokens   = query.lower().split()
      bm25_scores = bm25.get_scores(tokens)
      top_idx  = np.argsort(bm25_scores)[::-1][:top_k * 2]
      bm25_results = [
        {"text": chunks[i]["text"], "metadata": chunks[i]["metadata"], "score": float(bm25_scores[i])}
        for i in top_idx
      ]

    # Dense
    with timer (f"{prefix}dense", timings or {}):
      qvec    = embedder.encode([query])[0].tolist()
      results = collection.query(
          query_embeddings=[qvec],
          n_results=top_k * 2,
          include=["documents", "metadatas", "distances"]
      )
      dense_results = [
        {"text": d, "metadata": m, "score": 1 - s}
        for d, m, s in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        )
      ]

    # RRF fusion
    with timer (f"{prefix}rrf", timings or {}):
      rrf_scores = {}
      for rank, r in enumerate(dense_results):
          key = r["text"][:50]
          rrf_scores[key] = rrf_scores.get(key, {"score": 0, "data": r})
          rrf_scores[key]["score"] += 0.5 * (1 / (rank + 1))
      for rank, r in enumerate(bm25_results):
          key = r["text"][:50]
          if key not in rrf_scores:
              rrf_scores[key] = {"score": 0, "data": r}
          rrf_scores[key]["score"] += 0.5 * (1 / (rank + 1))
      rrf_results = [r["data"] for r in sorted(
          rrf_scores.values(), key=lambda x: x["score"], reverse=True
      )][:top_k]

    return {
        "bm25":   bm25_results[:top_k],
        "dense":  dense_results[:top_k],
        "rrf":    rrf_results,
    }

In [ ]:
def generate(query, chunks_for_generation, max_new_tokens=300):
    """
    Takes a query and a list of chunks,
    builds prompt, runs Mistral, returns answer string.
    """
    if not chunks_for_generation:
        return "No relevant chunks found — cannot generate answer."

    prompt    = build_prompt(query, chunks_for_generation)
    inputs    = tokenizer(prompt, return_tensors="pt").to(model_llm.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model_llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][input_len:]
    answer     = tokenizer.decode(new_tokens, skip_special_tokens=True)
    if "Technician's question:" in answer:
        answer = answer[:answer.index("Technician's question:")].strip()
    return answer.strip()

aggiungere ultime modifiche a synonym map

In [ ]:
SYNONYM_MAP = {
    # --- COMPONENTS ---
    "gearbox":              ["transmission", "dualogic", "gearbox fault light"],
    "gear box":             ["transmission", "dualogic", "gearbox fault light"],
    "gear with exclamation":["transmission", "dualogic", "gearbox fault light"],
    "parking brake":        ["handbrake"],
    "boot":                 ["luggage compartment"],
    "trunk":                ["luggage compartment"],
    "blinker":              ["direction indicator"],
    "turn signal":          ["direction indicator"],
    "ac":                   ["air conditioning", "climate control"],
    "air con":              ["air conditioning", "climate control"],
    "aircon":               ["air conditioning", "climate control"],
    "rev counter":          ["tachometer"],
    "rpm gauge":            ["tachometer"],
    "shift":                ["gearbox"],

    # --- SYMPTOMS ---
    "won't start":          ["engine starting failure"],
    "wont start":           ["engine starting failure"],
    "doesn't start":        ["engine starting failure"],
    "doesnt start":         ["engine starting failure"],
    "not starting":         ["engine starting failure"],
    "can't start":          ["engine starting failure"],
    "cant start":           ["engine starting failure"],
    "overheating":          ["engine coolant temperature"],
    "getting hot":          ["engine coolant temperature"],
    "too hot":              ["engine coolant temperature"],
    "running hot":          ["engine coolant temperature"],
    "making noise":         ["abnormal noise"],
    "strange noise":        ["abnormal noise"],
    "weird noise":          ["abnormal noise"],
    "pulling to one side":  ["steering"],
    "drifting":             ["steering"],
    "turning":             ["steering"],
    "drinking fuel":        ["fuel consumption"],
    "using too much fuel":  ["fuel consumption"],
    "high fuel use":        ["fuel consumption"],
    "stalling":             ["engine stalling", "ignition malfunction", "Fuel cut-off"],
    "cutting out":          ["engine stalling", "ignition malfunction", "Fuel cut-off"],
    "dying":                ["engine stalling", "ignition malfunction", "Fuel cut-off"],
    "squeaking":            ["vibration", "noise", "pulsating"],
    "squealing":            ["vibration", "noise", "pulsating"],
    "shaking":              ["vibration", "noise", "pulsating"],
    "shuddering":           ["vibration", "noise", "pulsating"],
    "vibrating":            ["vibration", "noise", "pulsating"],
    "flat tyre":            ["tyre pressure", "tyre"],
    "puncture":             ["tyre pressure", "tyre"],
    "blowout":              ["tyre pressure", "tyre"],
    "tyres wearing":        ["tyre pressure", "tyre"],
    "wearing unevenly":     ["tyre pressure", "tyre"],
    "smoke":                ["engine smoke", "overheating", "cooling", "emergency"],
    "smoking":              ["engine smoke", "overheating", "cooling", "emergency"],
    "grinding":             ["brake pad wear"],
    "smell":                ["leak", "overheating"],
    "not working":          ["replacing", "replacement"],
    "leak":                 ["check fluid"],

    # --- ICON VISUAL DESCRIPTIONS ---
    "spanner":                                   ["scheduled servicing"],
    "wrench symbol":                             ["scheduled servicing"],
    "tool icon":                                 ["scheduled servicing"],
    "handbrake warning":                         ["handbrake"],
    "oil can":                                   ["insufficient engine oil pressure light"],
    "oil symbol":                                ["insufficient engine oil pressure light"],
    "oil drop":                                  ["insufficient engine oil pressure light"],
    "oil light":                                 ["insufficient engine oil pressure light"],
    "oil can with waves":                        ["engine oil minimum level light"],
    "oil drop over water":                       ["engine oil minimum level light"],
    "oil symbol with waves":                     ["engine oil minimum level light"],
    "exclamation mark in triangle":              ["general failure warning light"],
    "warning triangle":                          ["general failure warning light"],
    "triangle with exclamation":                 ["general failure warning light"],
    "exclamation mark in circle":                ["low brake fluid handbrake engaged light"],
    "circle with exclamation":                   ["low brake fluid handbrake engaged light"],
    "round exclamation":                         ["low brake fluid handbrake engaged light"],
    "seatbelt":                                  ["seat belts not fastened light"],
    "belt symbol":                               ["seat belts not fastened light"],
    "car with lock":                             ["fiat code system failure light"],
    "car with padlock":                          ["fiat code system failure light"],
    "car with a padlock symbol":                 ["fiat code system failure light"],
    "snowflake":                                 ["possible ice on road light"],
    "ice symbol":                                ["possible ice on road light"],
    "ice icon":                                  ["possible ice on road light"],
    "abs circle exclamation":                    ["EBD failure light"],
    "steering wheel exclamation":                ["dualdrive electric power steering system failure"],
    "wheel with exclamation":                    ["dualdrive electric power steering system failure"],
    "car going uphill":                          ["hill holder system failure"],
    "car in circle going up":                    ["hill holder system failure"],
    "filter with smoke":                         ["DPF cleaning particulate trap"],
    "particles blowing through":                 ["DPF cleaning particulate trap"],
    "air blowing through box":                   ["DPF cleaning particulate trap"],
    "p with radar waves":                        ["parking sensor failure light"],
    "p with signal waves":                       ["parking sensor failure light"],
    "huge p with wifi":                          ["parking sensor failure light"],
    "fuel pump with drops":                      ["water in diesel filter"],
    "gas pump with liquid dripping":             ["water in diesel filter"],
    "light bulb exclamation":                    ["external lights failure"],
    "circle with dots":                          ["brake pad wear"],
    "horseshoe with exclamation":                ["tyre pressure warning itpms"],
    "horseshoe shape with an exclamation mark":  ["tyre pressure warning itpms"],
    "flat tire light":                           ["tyre pressure warning itpms"],
    "bowl with exclamation":                     ["tyre pressure warning itpms"],
    "person with circle x":                      ["passenger side airbag deactivated"],
    "airbag light two":                          ["passenger side airbag deactivated"],
    "passenger airbag":                          ["passenger side airbag deactivated"],
    "light with vertical line":                  ["rear fog light"],
    "light lines left vertical":                 ["front fog lights"],
    "speedometer icon":                          ["cruise control"],
    "dial with needle":                          ["cruise control"],
    "two headlights":                            ["side lights dipped beam headlights"],
    "headlight going left":                      ["main beam headlights"],
    "high beam":                                 ["main beam headlights"],
    "lpg failure":                               ["lpg system failure"],
    "fuel pump lpg":                             ["lpg system failure"],
    "cng":                                       ["methane system failure"],
    "methane":                                   ["methane system failure"],
    "check engine light":                        ["injection eobd system failure"],
    "engine light":                              ["injection eobd system failure"],
    "engine warning":                            ["injection eobd system failure"],
    "engine shaped light":                       ["injection eobd system failure"],
    "abs light":                                 ["abs failure light"],
    "abs written":                               ["abs failure light"],
    "airbag light":                              ["air bag failure"],
    "person with circle":                        ["air bag failure"],
    "car with skid marks":                       ["esc system failure"],
    "blue headlight":                            ["main beam headlights"],
}

In [ ]:
def normalize_apostrophes(text):
    return text.replace('\u2019', "'").replace('\u2018', "'")

def clean_query(text):
    text    = normalize_apostrophes(text.lower().strip())
    fillers = ["um", "uh", "like", "you know", "i mean", "so", "basically"]
    for f in fillers:
        text = re.sub(rf'\b{f}\b', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def enrich_query(query):
    cleaned    = clean_query(query)
    expansions = []
    matched    = []
    for colloquial, technical_terms in SYNONYM_MAP.items():
        if normalize_apostrophes(colloquial) in cleaned:
            expansions.extend(technical_terms)
            matched.append(colloquial)
    if expansions:
        seen     = set()
        unique   = [x for x in expansions if not (x in seen or seen.add(x))]
        enriched = cleaned + " " + " ".join(unique)
    else:
        enriched = cleaned
    return enriched, matched

In [ ]:
current_vehicle = None
chunks          = []
collection      = None
bm25            = None

def load_vehicle(vehicle):
    global current_vehicle, chunks, collection, bm25

    chunks_path = f"{CHUNKS_DIR}/{vehicle}.json"
    with open(chunks_path, encoding="utf-8") as f:
        chunks = json.load(f)
    print(f"Loaded {len(chunks)} chunks")

    tokenized = [c["text"].lower().split() for c in chunks]
    bm25      = BM25Okapi(tokenized)
    print(f"BM25 ready — {len(tokenized)} documents")

    collection = client.get_collection(vehicle)
    print(f"ChromaDB loaded — {collection.count()} chunks")

    current_vehicle = vehicle
    print(f"Vehicle set to: {current_vehicle}")

def check_vehicle():
    global current_vehicle
    if current_vehicle is not None:
        return True
    vehicle = input("Please specify your vehicle (e.g. 'punto'): ").strip()
    if vehicle:
        load_vehicle(vehicle)
        return True
    print("No vehicle specified — cannot proceed.")
    return False

In [ ]:
def speak(text, speed=150, save_path=None):
    path = save_path or "/content/drive/MyDrive/thesis/result.wav"

    subprocess.run([
        "espeak",
        "-v", "en",
        "-s", str(speed),
        "-w", path,
        text
    ], check=True)

    display(Audio(path, autoplay=True))
    return path

In [ ]:
def print_results(label, results_dict, show_scores=True):
    """
    Standardised printing for all retrieval stages.
    Pass label (e.g. "RAW QUERY") and results dict from retrieve().
    """
    print(f"\n")
    print(f"  {label}")
    print(f"\n")

    for stage in ["bm25", "dense", "rrf", "reranked"]:
        if stage not in results_dict:
            continue
        print(f"\n--- {stage.upper()} ---")
        for i, r in enumerate(results_dict[stage]):
            page  = r["metadata"].get("page_number", "?")
            score = r.get("rerank_score", r.get("score", 0))
            score_str = f"| score {score:.3f} " if show_scores else ""
            print(f"  [{i+1}] p{page} {score_str}| {r['text'][:100]}...")

def print_answer(label, answer):
    print(f"\n")
    print(f"  ANSWER — {label}")
    print(f"\n")
    print(answer)

In [ ]:
def pipeline(
    query         = None,
    audio_path    = None,
    use_enrichment = True,    # False → skip enrichment, use raw query only
    generate_answers = True,  # False → skip generation, show retrieval only
    speak_answer  = True,
    top_k         = 5,
    measure_latency  = True,
):
    """
    Full pipeline with experimental flags:
      use_enrichment=False   → retrieval on raw query only (no SYNONYM_MAP)
      generate_answers=False → skip Mistral, useful for fast retrieval testing
      speak_answer=False     → skip TTS
    """
    if not check_vehicle():
        return

    timings = {}

    # Input
    if audio_path is not None:
        with timer("asr", timings):
          result    = model_asr.transcribe(
            audio_path,
            language="en",
            initial_prompt="Automotive diagnostic report. Car problems, warning lights, engine issues.",
            fp16=True,
          )
        raw_query = result["text"].strip()
        print(f"Transcript:  {raw_query}")
    elif query is not None:
        raw_query = query
    else:
        print("ERROR: provide either query= or audio_path=")
        return

    # Query analysis
    with timer("query_analysis", timings):
      enriched, matched = enrich_query(raw_query)

    print(f"\nORIGINAL:  {raw_query}")
    if use_enrichment and matched:
        print(f"ENRICHED:  {enriched}")
        print(f"TRIGGERED: {matched}")
    elif use_enrichment and not matched:
        print("ENRICHED:  (no enrichment applied — query already technical)")
    else:
        print("ENRICHED:  (enrichment disabled)")

    # Retrieval

    with timer("retrieval_raw", timings):
        raw_results = retrieve_all(raw_query, top_k=top_k, timings=timings, prefix="raw_")
        raw_reranked            = rerank(raw_query, raw_results["rrf"])[:top_k]
        raw_results["reranked"] = raw_reranked
    print_results("RETRIEVAL — RAW QUERY", raw_results)

    if use_enrichment and matched:
        with timer("retrieval_enriched", timings):
            enriched_results = retrieve_all(enriched, top_k=top_k, timings=timings, prefix="enriched_")
            enriched_reranked            = rerank(enriched, enriched_results["rrf"])[:top_k]
            enriched_results["reranked"] = enriched_reranked
        print_results("RETRIEVAL — ENRICHED QUERY", enriched_results)
    else:
        enriched_results = raw_results


    # Generation
    if generate_answers:
        with timer("generation_rrf_raw", timings):
          # Condition 1: RRF only, raw query
          ans_rrf_raw = generate(raw_query, raw_results["rrf"])
        print_answer("RRF | raw query", ans_rrf_raw)

        # Condition 2: Reranked, raw query
        with timer("generation_ranked_raw", timings):
          ans_reranked_raw = generate(raw_query, raw_results["reranked"])
        print_answer("Reranked | raw query", ans_reranked_raw)

        if use_enrichment and matched:
            with timer("generation_rrf_enriched", timings):
              # Condition 3: RRF only, enriched query
              ans_rrf_enriched = generate(enriched, enriched_results["rrf"])
            print_answer("RRF | enriched query", ans_rrf_enriched)

            with timer("generation_ranked_enriched", timings):
              # Condition 4: Reranked, enriched query
              ans_reranked_enriched = generate(enriched, enriched_results["reranked"])
            print_answer("Reranked | enriched query", ans_reranked_enriched)

            # TTS on best answer (reranked + enriched)
            if speak_answer:
                print("\n Speaking best answer (reranked + enriched)...")
                speak(ans_reranked_enriched)
        else:
            if speak_answer:
                print("\n Speaking best answer (reranked)...")
                speak(ans_reranked_raw)
    else:
        print("\n(generation skipped — set generate_answers=True to run Mistral)")

    if measure_latency:
        print_timings(timings)

    return timings

Latency test

In [ ]:
def run_evaluation(
    json_path,
    output_path,
    top_k=5,
):
    if not check_vehicle():
        return

    with open(json_path, encoding="utf-8") as f:
        dataset = json.load(f)

    if os.path.exists(output_path):
        with open(output_path, encoding="utf-8") as f:
            results = json.load(f)
    else:
        results = {}

    total = len(dataset)

    for idx, entry in enumerate(dataset):
        query_key = f"query_{idx + 1}"

        if query_key in results:
            continue

        user_input = entry["user_input"]

        t0 = time.perf_counter()
        enriched, triggered = enrich_query(user_input)
        enrich_latency = time.perf_counter() - t0

        # RAW pipeline

        t0 = time.perf_counter()
        raw_rrf = retrieve_all(user_input, top_k=top_k)
        raw_retrieval_latency = time.perf_counter() - t0

        t0 = time.perf_counter()
        raw_ans = generate(user_input, raw_rrf["rrf"])
        raw_generation_latency = time.perf_counter() - t0

        raw_pages = [
            r["metadata"].get("page_number")
            for r in raw_rrf["rrf"]
        ]

        # ENRICHED pipeline

        t0 = time.perf_counter()
        enr_rrf = retrieve_all(enriched, top_k=top_k)
        enr_retrieval_latency = time.perf_counter() - t0

        t0 = time.perf_counter()
        enr_ans = generate(enriched, enr_rrf["rrf"])
        enr_generation_latency = time.perf_counter() - t0

        enr_pages = [
            r["metadata"].get("page_number")
            for r in enr_rrf["rrf"]
        ]

        # RERANKED pipeline

        t0 = time.perf_counter()
        enr_rrf_wide = retrieve_all(enriched, top_k=top_k * 2)
        reranked_chunks = rerank(
            enriched,
            enr_rrf_wide["rrf"]
        )[:top_k]
        rerank_latency = time.perf_counter() - t0

        t0 = time.perf_counter()
        rer_ans = generate(enriched, reranked_chunks)
        rer_generation_latency = time.perf_counter() - t0

        rer_pages = [
            r["metadata"].get("page_number")
            for r in reranked_chunks
        ]

        rer_scores = [
            round(r.get("rerank_score", 0), 3)
            for r in reranked_chunks
        ]

        results[query_key] = {
            "user_input": user_input,
            "category": entry.get("category", ""),
            "relevant_pages": entry.get("relevant_pages", []),

            "enriched_query": enriched,
            "triggered": triggered,

            "latency": {
                "query_enrichment": enrich_latency,

                "raw_retrieval": raw_retrieval_latency,
                "raw_generation": raw_generation_latency,

                "enriched_retrieval": enr_retrieval_latency,
                "enriched_generation": enr_generation_latency,

                "reranking": rerank_latency,
                "reranked_generation": rer_generation_latency,
            },

            "raw": {
                "retrieved_pages": raw_pages,
                "answer": raw_ans,
            },

            "enriched": {
                "retrieved_pages": enr_pages,
                "answer": enr_ans,
            },

            "reranked": {
                "retrieved_pages": rer_pages,
                "rerank_scores": rer_scores,
                "answer": rer_ans,
            },
        }

        # Save after every query
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

    return results

In [ ]:
run_evaluation(
    json_path="/content/drive/MyDrive/ground_truth.json",
    output_path="/content/drive/MyDrive/latency_results.json",
    top_k=5
)